# OntologySim - Silver Transformation

Source-level normalization of Bronze engineering data.

Silver normalizes structure and values, but does not yet resolve cross-source identities.

In [0]:
from pyspark.sql import functions as F

generation_id = "8573dcf32f2ed486"
profile = "medium"


def write_generation(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_generation_id = '{generation_id}'",
        )
        .saveAsTable(table_name)
    )

## Test Management

In [0]:
bronze_test_cases_df = (
    spark.table("ontologysim.bronze.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

silver_test_cases_df = (
    bronze_test_cases_df
    .select(
        F.trim("test_case_id").alias("test_case_id"),
        F.trim("title").alias("title"),
        F.trim("description").alias("description"),

        F.upper(F.trim("test_type")).alias("test_type"),
        F.trim("preconditions").alias("preconditions"),
        F.trim("test_steps").alias("test_steps"),
        F.trim("expected_result").alias("expected_result"),

        F.upper(F.trim("priority")).alias("priority"),
        F.upper(F.trim("status")).alias("status"),

        F.trim("owner_team").alias("owner_team"),
        F.trim("version").alias("version"),

        F.trim("requirement_refs").alias("requirement_refs"),
        F.trim("component_refs").alias("component_refs"),

        F.upper(F.trim("target_ecu")).alias("target_ecu"),

        F.to_timestamp(
            "created_at",
            "yyyy-MM-dd HH:mm:ss",
        ).alias("created_at"),

        F.to_timestamp(
            "updated_at",
            "yyyy-MM-dd HH:mm:ss",
        ).alias("updated_at"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_test_cases_df,
    "ontologysim.silver.test_cases",
)

In [0]:
bronze_rows = bronze_test_cases_df.count()

silver_df = (
    spark.table("ontologysim.silver.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    silver_df
    .filter(
        F.col("created_at").isNull()
        | F.col("updated_at").isNull()
    )
    .count()
)

print(f"Bronze rows:             {bronze_rows}")
print(f"Silver rows:             {silver_rows}")
print(f"Timestamp parse failures:{timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("Test Management Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("Test Management Silver validation passed")

Bronze rows:             16080
Silver rows:             16080
Timestamp parse failures:0
Test Management Silver validation passed


## Test Execution

In [0]:
bronze_test_executions_df = (
    spark.table("ontologysim.bronze.test_executions")
    .filter(F.col("_generation_id") == generation_id)
)

silver_test_executions_df = (
    bronze_test_executions_df
    .select(
        F.trim("execution_id").alias("execution_id"),
        F.trim("test_case_id").alias("test_case_id"),
        F.trim("campaign_id").alias("campaign_id"),
        F.trim("session_id").alias("session_id"),

        F.to_timestamp("start_timestamp").alias("start_timestamp"),
        F.to_timestamp("end_timestamp").alias("end_timestamp"),

        F.col("duration_ms").cast("long").alias("duration_ms"),
        F.upper(F.trim("result")).alias("result"),

        F.trim("execution_environment").alias("execution_environment"),
        F.trim("vehicle_id").alias("vehicle_id"),

        F.trim("software_version_hcp3").alias("software_version_hcp3"),
        F.trim("software_version_hcp5").alias("software_version_hcp5"),
        F.trim("software_version_conmod").alias("software_version_conmod"),

        F.trim("hardware_configuration").alias("hardware_configuration"),
        F.trim("executor").alias("executor"),
        F.trim("failure_reason").alias("failure_reason"),

        F.transform("log_refs", lambda x: F.trim(x)).alias("log_refs"),
        F.transform("measurement_refs", lambda x: F.trim(x)).alias(
            "measurement_refs"
        ),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_test_executions_df,
    "ontologysim.silver.test_executions",
)

In [0]:
bronze_rows = bronze_test_executions_df.count()

silver_df = (
    spark.table("ontologysim.silver.test_executions")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_test_executions_df
    .filter(
        (
            F.col("start_timestamp").isNotNull()
            & F.to_timestamp("start_timestamp").isNull()
        )
        | (
            F.col("end_timestamp").isNotNull()
            & F.to_timestamp("end_timestamp").isNull()
        )
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("Test Execution Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("Test Execution Silver validation passed")

Bronze rows:              160800
Silver rows:              160800
Timestamp parse failures: 0
Test Execution Silver validation passed


## Requirements

In [0]:
bronze_requirements_df = (
    spark.table("ontologysim.bronze.requirements")
    .filter(F.col("_generation_id") == generation_id)
)

silver_requirements_df = (
    bronze_requirements_df
    .select(
        F.trim("REQUIREMENT-ID").alias("requirement_id"),
        F.trim("TITLE").alias("title"),
        F.trim("DESCRIPTION").alias("description"),

        F.upper(F.trim("REQUIREMENT-TYPE")).alias("requirement_type"),
        F.upper(F.trim("PRIORITY")).alias("priority"),
        F.upper(F.trim("STATUS")).alias("status"),

        F.trim("OWNER-TEAM").alias("owner_team"),
        F.trim("VERSION").alias("version"),

        F.to_timestamp("CREATED-AT").alias("created_at"),
        F.to_timestamp("UPDATED-AT").alias("updated_at"),

        F.trim("PARENT-REQUIREMENT-ID").alias("parent_requirement_id"),
        F.trim("RELATED-REQUIREMENT-IDS").alias("related_requirement_ids"),

        F.upper(F.trim("TARGET-ECU")).alias("target_ecu"),
        F.trim("TARGET-COMPONENT").alias("target_component"),
        F.upper(F.trim("VERIFICATION-METHOD")).alias("verification_method"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_requirements_df,
    "ontologysim.silver.requirements",
)

In [0]:
bronze_rows = bronze_requirements_df.count()

silver_df = (
    spark.table("ontologysim.silver.requirements")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_requirements_df
    .filter(
        (
            F.col("CREATED-AT").isNotNull()
            & F.to_timestamp("CREATED-AT").isNull()
        )
        | (
            F.col("UPDATED-AT").isNotNull()
            & F.to_timestamp("UPDATED-AT").isNull()
        )
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("Requirements Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("Requirements Silver validation passed")

Bronze rows:              8040
Silver rows:              8040
Timestamp parse failures: 0
Requirements Silver validation passed


## Architecture

In [0]:
bronze_components_df = (
    spark.table("ontologysim.bronze.architecture_components")
    .filter(F.col("_generation_id") == generation_id)
)

silver_components_df = (
    bronze_components_df
    .select(
        F.trim("COMPONENT-ID").alias("component_id"),
        F.trim("COMPONENT-NAME").alias("component_name"),
        F.upper(F.trim("COMPONENT-TYPE")).alias("component_type"),
        F.trim("ECU-ID").alias("ecu_id"),
        F.trim("RESPONSIBILITY").alias("responsibility"),
        F.trim("VERSION").alias("version"),
        F.upper(F.trim("STATUS")).alias("status"),
        F.trim("OWNER-TEAM").alias("owner_team"),
        F.trim("PARENT-COMPONENT-ID").alias("parent_component_id"),

        F.trim("PROVIDED-INTERFACES").alias("provided_interfaces"),
        F.trim("REQUIRED-INTERFACES").alias("required_interfaces"),
        F.trim("PROVIDED-SERVICES").alias("provided_services"),
        F.trim("REQUIRED-SERVICES").alias("required_services"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_components_df,
    "ontologysim.silver.architecture_components",
)

In [0]:
bronze_ecus_df = (
    spark.table("ontologysim.bronze.architecture_ecus")
    .filter(F.col("_generation_id") == generation_id)
)

silver_ecus_df = (
    bronze_ecus_df
    .select(
        F.trim("ECU-ID").alias("ecu_id"),
        F.trim("ECU-NAME").alias("ecu_name"),
        F.trim("ROLE").alias("role"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_ecus_df,
    "ontologysim.silver.architecture_ecus",
)

In [0]:
bronze_component_rows = bronze_components_df.count()
silver_component_rows = (
    spark.table("ontologysim.silver.architecture_components")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

bronze_ecu_rows = bronze_ecus_df.count()
silver_ecu_rows = (
    spark.table("ontologysim.silver.architecture_ecus")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Components — Bronze: {bronze_component_rows}, Silver: {silver_component_rows}")
print(f"ECUs       — Bronze: {bronze_ecu_rows}, Silver: {silver_ecu_rows}")

if bronze_component_rows != silver_component_rows:
    raise ValueError("Architecture component row count mismatch")

if bronze_ecu_rows != silver_ecu_rows:
    raise ValueError("Architecture ECU row count mismatch")

print("Architecture Silver validation passed")

Components — Bronze: 1600, Silver: 1600
ECUs       — Bronze: 3, Silver: 3
Architecture Silver validation passed


## Defects / KPM

In [0]:
bronze_defects_df = (
    spark.table("ontologysim.bronze.defects")
    .filter(F.col("_generation_id") == generation_id)
)

silver_defects_df = (
    bronze_defects_df
    .select(
        F.trim("defect_id").alias("defect_id"),
        F.trim("title").alias("title"),
        F.trim("description").alias("description"),

        F.upper(F.trim("severity")).alias("severity"),
        F.upper(F.trim("priority")).alias("priority"),
        F.upper(F.trim("status")).alias("status"),

        F.trim("responsible_team").alias("responsible_team"),
        F.trim("reporter").alias("reporter"),

        F.to_timestamp(
            "created_at",
            "yyyy-MM-dd HH:mm:ss",
        ).alias("created_at"),

        F.to_timestamp(
            "updated_at",
            "yyyy-MM-dd HH:mm:ss",
        ).alias("updated_at"),

        F.upper(F.trim("affected_ecu")).alias("affected_ecu"),
        F.trim("affected_component").alias("affected_component"),
        F.trim("affected_software_version").alias("affected_software_version"),

        F.trim("requirement_refs").alias("requirement_refs"),
        F.trim("test_case_refs").alias("test_case_refs"),
        F.trim("test_execution_refs").alias("test_execution_refs"),

        F.trim("resolution").alias("resolution"),
        F.trim("root_cause").alias("root_cause"),
        F.trim("duplicate_of").alias("duplicate_of"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_defects_df,
    "ontologysim.silver.defects",
)

In [0]:
bronze_rows = bronze_defects_df.count()

silver_df = (
    spark.table("ontologysim.silver.defects")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_defects_df
    .filter(
        (
            F.col("created_at").isNotNull()
            & F.to_timestamp(
                "created_at",
                "yyyy-MM-dd HH:mm:ss",
            ).isNull()
        )
        | (
            F.col("updated_at").isNotNull()
            & F.to_timestamp(
                "updated_at",
                "yyyy-MM-dd HH:mm:ss",
            ).isNull()
        )
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("Defects Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("Defects Silver validation passed")

Bronze rows:              3216
Silver rows:              3216
Timestamp parse failures: 0
Defects Silver validation passed


## Trace Export

In [0]:
bronze_traces_df = (
    spark.table("ontologysim.bronze.traces")
    .filter(F.col("_generation_id") == generation_id)
)

silver_traces_df = (
    bronze_traces_df
    .select(
        F.trim("trace_id").alias("trace_id"),
        F.upper(F.trim("source_type")).alias("source_type"),
        F.trim("source_id").alias("source_id"),
        F.upper(F.trim("relationship_type")).alias("relationship_type"),
        F.upper(F.trim("target_type")).alias("target_type"),
        F.trim("target_id").alias("target_id"),
        F.upper(F.trim("trace_status")).alias("trace_status"),

        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("updated_at").alias("updated_at"),

        F.trim("source_system").alias("source_system"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_traces_df,
    "ontologysim.silver.traces",
)

In [0]:
bronze_rows = bronze_traces_df.count()

silver_df = (
    spark.table("ontologysim.silver.traces")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_traces_df
    .filter(
        (
            F.col("created_at").isNotNull()
            & F.to_timestamp("created_at").isNull()
        )
        | (
            F.col("updated_at").isNotNull()
            & F.to_timestamp("updated_at").isNull()
        )
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("Traces Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("Traces Silver validation passed")

Bronze rows:              155976
Silver rows:              155976
Timestamp parse failures: 0
Traces Silver validation passed


## ConMod Runtime

In [0]:
bronze_conmod_df = (
    spark.table("ontologysim.bronze.conmod_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

silver_conmod_df = (
    bronze_conmod_df
    .select(
        F.to_timestamp("timestamp").alias("timestamp"),

        F.upper(F.trim("ecu")).alias("ecu"),
        F.trim("service").alias("service"),
        F.upper(F.trim("event")).alias("event"),
        F.upper(F.trim("severity")).alias("severity"),

        F.upper(F.trim("network_type")).alias("network_type"),
        F.upper(F.trim("connection_state")).alias("connection_state"),
        F.col("signal_strength").cast("long").alias("signal_strength"),
        F.upper(F.trim("backend_state")).alias("backend_state"),

        F.trim("session_id").alias("session_id"),
        F.trim("execution_ref").alias("execution_ref"),
        F.trim("software_build").alias("software_build"),
        F.trim("message").alias("message"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_conmod_df,
    "ontologysim.silver.conmod_runtime",
)

In [0]:
bronze_rows = bronze_conmod_df.count()

silver_df = (
    spark.table("ontologysim.silver.conmod_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_conmod_df
    .filter(
        F.col("timestamp").isNotNull()
        & F.to_timestamp("timestamp").isNull()
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("ConMod Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("ConMod Silver validation passed")

Bronze rows:              16497867
Silver rows:              16497867
Timestamp parse failures: 0
ConMod Silver validation passed


## HCP5 Runtime

In [0]:
bronze_hcp5_df = (
    spark.table("ontologysim.bronze.hcp5_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

silver_hcp5_df = (
    bronze_hcp5_df
    .select(
        F.to_timestamp(
            "timestamp",
            "dd.MM.yyyy HH:mm:ss.SSS",
        ).alias("timestamp"),

        F.upper(F.trim("severity")).alias("severity"),
        F.upper(F.trim("ecu")).alias("ecu"),
        F.trim("component").alias("component"),
        F.upper(F.trim("event")).alias("event"),

        F.trim("session_id").alias("session_id"),
        F.trim("execution_id").alias("execution_id"),
        F.trim("software_version").alias("software_version"),
        F.trim("message").alias("message"),

        "_raw_line",
        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_hcp5_df,
    "ontologysim.silver.hcp5_runtime",
)

In [0]:
bronze_rows = bronze_hcp5_df.count()

silver_df = (
    spark.table("ontologysim.silver.hcp5_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_hcp5_df
    .filter(
        F.col("timestamp").isNotNull()
        & F.to_timestamp(
            "timestamp",
            "dd.MM.yyyy HH:mm:ss.SSS",
        ).isNull()
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("HCP5 Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("HCP5 Silver validation passed")

Bronze rows:              11406745
Silver rows:              11406745
Timestamp parse failures: 0
HCP5 Silver validation passed


## HCP3 Runtime

In [0]:
bronze_hcp3_df = (
    spark.table("ontologysim.bronze.hcp3_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

silver_hcp3_df = (
    bronze_hcp3_df
    .select(
        F.to_timestamp(
            "timestamp",
            "yyyy-MM-dd HH:mm:ss.SSS",
        ).alias("timestamp"),

        F.trim("ecu_id").alias("ecu_id"),
        F.trim("application_id").alias("application_id"),
        F.trim("context_id").alias("context_id"),
        F.trim("component_name").alias("component_name"),

        F.upper(F.trim("severity")).alias("severity"),
        F.upper(F.trim("event_type")).alias("event_type"),

        F.trim("message").alias("message"),
        F.trim("session_id").alias("session_id"),
        F.trim("execution_id").alias("execution_id"),
        F.trim("software_version").alias("software_version"),
        F.trim("thread_id").alias("thread_id"),

        "_raw_line",
        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_hcp3_df,
    "ontologysim.silver.hcp3_runtime",
)

In [0]:
bronze_rows = bronze_hcp3_df.count()

silver_df = (
    spark.table("ontologysim.silver.hcp3_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_hcp3_df
    .filter(
        F.col("timestamp").isNotNull()
        & F.to_timestamp(
            "timestamp",
            "yyyy-MM-dd HH:mm:ss.SSS",
        ).isNull()
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("HCP3 Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("HCP3 Silver validation passed")

Bronze rows:              10643872
Silver rows:              10643872
Timestamp parse failures: 0
HCP3 Silver validation passed


## Measurements

In [0]:
bronze_measurements_df = (
    spark.table("ontologysim.bronze.measurements")
    .filter(F.col("_generation_id") == generation_id)
)

silver_measurements_df = (
    bronze_measurements_df
    .select(
        F.trim("measurement_id").alias("measurement_id"),
        F.to_timestamp("timestamp_utc").alias("timestamp_utc"),
        F.trim("execution_ref").alias("execution_ref"),
        F.trim("session_ref").alias("session_ref"),
        F.upper(F.trim("ecu_id")).alias("ecu_id"),
        F.trim("component_ref").alias("component_ref"),
        F.trim("signal_name").alias("signal_name"),
        F.col("value_numeric").cast("double").alias("value_numeric"),
        F.trim("value_text").alias("value_text"),
        F.trim("unit").alias("unit"),

        "_source_file",
        "_generation_id",
        "_profile",
        "_source_family",
        "_ingestion_run_id",
        "_ingestion_timestamp",
    )
    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp(),
    )
)

write_generation(
    silver_measurements_df,
    "ontologysim.silver.measurements",
)

In [0]:
bronze_rows = bronze_measurements_df.count()

silver_df = (
    spark.table("ontologysim.silver.measurements")
    .filter(F.col("_generation_id") == generation_id)
)

silver_rows = silver_df.count()

timestamp_failures = (
    bronze_measurements_df
    .filter(
        F.col("timestamp_utc").isNotNull()
        & F.to_timestamp("timestamp_utc").isNull()
    )
    .count()
)

print(f"Bronze rows:              {bronze_rows}")
print(f"Silver rows:              {silver_rows}")
print(f"Timestamp parse failures: {timestamp_failures}")

if bronze_rows != silver_rows:
    raise ValueError("Measurements Bronze/Silver row count mismatch")

if timestamp_failures:
    raise ValueError(
        f"{timestamp_failures} timestamp conversions failed"
    )

print("Measurements Silver validation passed")

Bronze rows:              7603360
Silver rows:              7603360
Timestamp parse failures: 0
Measurements Silver validation passed
